# Hito 1 · Línea base sobre su conjunto de evaluación

**Apoyo opcional.** Este notebook no es obligatorio ni define cómo debe ser su línea base: es un punto de partida para la sección 3 del Hito 1. Adáptenlo a su proyecto o usen otro código.

Hace tres cosas:

1. Lee su `cases.jsonl` y su corpus.
2. Ejecuta una línea base simple: una instrucción sin ejemplos para la extracción y todo el corpus en la instrucción para las respuestas con fuentes, como en el laboratorio 3. También calcula dos líneas base triviales, sin modelo: «siempre `null`» y «siempre sin evidencia».
3. Calcula las métricas, guarda las salidas en `eval/baseline/` y vuelve a calcular las métricas desde esos archivos, sin llamar al modelo, como hará el docente al revisar.

Las métricas de este notebook son ejemplos. Usen las de sus criterios de éxito: si su criterio es otro, cambien la función que lo calcula.

## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación. Las claves van en **Secretos** (icono de llave), como en los laboratorios: `GEMINI_API_KEY` o `GOOGLE_API_KEY` fuera de clase, y `CURSO_API_KEY` y `CURSO_BASE_URL` para el servidor del curso, que funciona solo durante las clases de los viernes.

In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # fuera de Colab puede no existir el paquete google
    IN_COLAB = False

if IN_COLAB:
    PACKAGE_REF = "hito1"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                           f"curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}"])
    print("Paquete instalado. Continúa con la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")

In [ ]:
import asyncio
import json
import os
import shutil
from pathlib import Path
from statistics import median

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from curso_ia import create_model, document_input, lab_resource, load_config, load_json, status, timed_run

# Empieza en "simulated" para revisar el circuito sin gastar consultas; cambia a "real" para la línea base.
MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE

# Modelo que responde, con la forma proveedor:modelo:
#   "google:gemini-3.5-flash-lite"  Gemini, gratis con tu clave de Gemini; para trabajar fuera de clase.
#   "curso:course-chat"             servidor del curso, solo durante las clases de los viernes.
#   "google:gemma-4-26b-a4b-it"     Gemma 4, gratis; segunda opción, a menudo lenta o saturada.
MODEL = "google:gemini-3.5-flash-lite"  # @param ["google:gemini-3.5-flash-lite", "curso:course-chat", "google:gemma-4-26b-a4b-it"]
os.environ["GENERATION_MODEL"] = MODEL

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "hito1"  # aquí van sus archivos: cases.jsonl, corpus/ y documents/
OUTPUT = ROOT / "eval/baseline"
config = load_config(ROOT)
model_ref = config.models["generation"]
print("Modo:", config.mode, "· Modelo:", model_ref)
display(pd.DataFrame([row for row in status(config) if row["role"] == "generation"]))

## 2. Cargar sus casos y su corpus

Con `USE_EXAMPLE = True` el notebook usa un ejemplo armado con los datos de los laboratorios 2 y 3, para que vean el formato y prueben el circuito. Para usar los suyos, ponlo en `False` y suban sus archivos en el panel **Archivos** de Colab, dentro de la carpeta `hito1/`, que crea la celda siguiente:

- `hito1/cases.jsonl`: su conjunto, con el formato del enunciado del Hito 1 (un caso por línea, con `id`, `task`, `input` y `expected`).
- `hito1/corpus/`: los documentos que consulta su sistema para responder, en texto o Markdown. El nombre del archivo sin extensión es el `doc_id` de sus citas: `pol-equipaje.md` se cita como `pol-equipaje`.
- `hito1/documents/`: solo si en algunos casos de extracción el `input` es el nombre de un archivo (por ejemplo, `reclamo-07.jpg`) en vez del texto.

Colab borra los archivos al cerrar la sesión: vuelvan a subirlos si se desconecta.

In [ ]:
USE_EXAMPLE = True  # False para usar sus archivos de hito1/

for folder in (DATA / "corpus", DATA / "documents"):
    folder.mkdir(parents=True, exist_ok=True)


def example_cases():
    """Casos de ejemplo con los textos del laboratorio 2 y las preguntas del laboratorio 3, en el formato del Hito 1."""
    lab02 = load_json(lab_resource(2, "cases.json"))
    rows = [{"id": case["id"], "task": "extract",
             "input": lab_resource(2, case["document"]).read_text(encoding="utf-8"),
             "expected": case["reference"], "category": "laboratorio 2", "note": case.get("annotation_note", "")}
            for case in lab02["cases"] if case["document"].endswith(".txt")]
    rows += [{"id": question["id"], "task": "answer", "input": question["question"],
              "expected": {"answer": question["expected_answer"], "evidence": question["expected_evidence"],
                           "answerable": question["answerable"]},
              "category": question["category"]}
             for question in load_json(lab_resource(3, "questions.json"))]
    return rows


def example_corpus():
    return {meta["doc_id"]: lab_resource(3, f"documents/{meta['file']}").read_text(encoding="utf-8")
            for meta in load_json(lab_resource(3, "documents.json"))}


def read_cases(path):
    """Leer cases.jsonl y avisar, con el número de línea, de lo que impide usar un caso."""
    rows, problems = [], []
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            problems.append(f"línea {number}: no es JSON válido ({error.msg})")
            continue
        missing_keys = [key for key in ("id", "task", "input", "expected") if key not in row]
        if missing_keys or row.get("task") not in ("extract", "answer"):
            problems.append(f"línea {number}: faltan {missing_keys} o task no es extract ni answer")
            continue
        rows.append(row)
    ids = [row["id"] for row in rows]
    problems += [f"id repetido: {case_id}" for case_id in sorted({i for i in ids if ids.count(i) > 1})]
    return rows, problems


if USE_EXAMPLE:
    cases, corpus = example_cases(), example_corpus()
else:
    if not (DATA / "cases.jsonl").exists():
        raise FileNotFoundError("Sube su cases.jsonl a la carpeta hito1/ del panel Archivos y vuelve a ejecutar.")
    cases, problems = read_cases(DATA / "cases.jsonl")
    for problem in problems:
        print("⚠", problem)
    corpus = {path.stem: path.read_text(encoding="utf-8") for path in sorted((DATA / "corpus").glob("*"))
              if path.suffix.lower() in {".md", ".txt"}}

extract_cases = [case for case in cases if case["task"] == "extract"]
answer_cases = [case for case in cases if case["task"] == "answer"]
print(f"Casos de extracción: {len(extract_cases)} · de respuesta con fuentes: {len(answer_cases)}")
print(f"Documentos del corpus: {len(corpus)} · unos {sum(len(text) for text in corpus.values()) // 4:,} tokens en total")

## 3. Extracción: esquema e instrucción sin ejemplos

El esquema son los campos de su `expected`. El de abajo es el de los laboratorios: **cámbienlo por el suyo**, con un tipo y una descripción por campo. La línea base es la instrucción más simple, sin ejemplos; las mejoras (ejemplos, reglas de anotación en la instrucción, otro modelo) son sus alternativas de la sección 4 del Hito 1.

In [ ]:
from typing import Literal


class Record(BaseModel):
    booking_code: str | None = Field(description="Código de la reserva afectada")
    requested_action: Literal["rebooking", "refund"] | None = Field(
        description="Lo que pide el pasajero: rebooking (cambio de vuelo) o refund (devolución)")
    amount_clp: int | None = Field(description="Monto pagado por el pasaje, en pesos chilenos")
    flight_date: str | None = Field(description="Fecha del vuelo en formato AAAA-MM-DD")
    delay_minutes: int | None = Field(description="Retraso del vuelo en minutos")


EXTRACT_INSTRUCTION = """
Extrae los campos del documento. Usa solo lo que dice el documento.
Si un dato no aparece o no puedes determinarlo con certeza, usa null.
""".strip()
SETTINGS = {"temperature": 1.0, "max_tokens": 1500}

FIELDS = list(Record.model_fields)
not_in_schema = sorted({key for case in extract_cases for key in case["expected"]} - set(FIELDS))
if not_in_schema:
    print("⚠ Campos de expected que no están en Record (no se miden):", not_in_schema)
print("Campos medidos:", FIELDS)

In [ ]:
# Límites gratuitos de AI Studio: consultas y tokens por minuto. Se espera entre consultas para no chocar con ellos.
LIMITS = {"google:gemma-4-26b-a4b-it": (30, 16_000), "google:gemini-3.5-flash-lite": (15, 250_000)}


def pause_after(row):
    if config.mode != "real" or model_ref not in LIMITS:
        return 0
    per_minute, tokens_per_minute = LIMITS[model_ref]
    return max(60 / per_minute, 60 * (row.get("input_tokens") or 0) / tokens_per_minute)


def extract_input(case):
    """El texto del caso o, si input es el nombre de un archivo de hito1/documents/, ese archivo."""
    path = DATA / "documents" / str(case["input"])
    if not USE_EXAMPLE and path.is_file():
        return ["Extrae los campos del documento adjunto.", document_input(path)]
    return f"Documento:\n{case['input']}"


async def extract(case):
    model = create_model(model_ref, config, simulated_output={field: None for field in FIELDS})
    agent = Agent(model, output_type=Record, instructions=EXTRACT_INSTRUCTION, model_settings=SETTINGS, retries=1)
    row = await timed_run(agent, extract_input(case), ref=model_ref, config=config)
    return {"id": case["id"], "task": "extract", **row}


extract_rows = []
for case in extract_cases:
    row = await extract(case)
    extract_rows.append(row)
    print(case["id"], row["status"], row.get("error_type", ""))
    await asyncio.sleep(pause_after(row))

## 4. Respuestas con fuentes: todo el corpus en la instrucción

Como en el laboratorio 3, la línea base envía todo el corpus en cada consulta. Si responde bien, la recuperación solo puede ganar en costo y latencia; si responde mal, el problema está en el generador o en el corpus, no en el recuperador. Si su corpus no cabe en la ventana del modelo (el servidor del curso admite unos 32.000 tokens), usen otra línea base, por ejemplo la recuperación del laboratorio 3 con su configuración más simple.

In [ ]:
class Citation(BaseModel):
    doc_id: str = Field(description="Identificador del documento citado, por ejemplo pol-equipaje")
    quote: str = Field(description="Frase copiada literalmente del documento que respalda la respuesta")


class Answer(BaseModel):
    answer: str = Field(description="Respuesta breve en español, sin citas ni identificadores de documentos")
    citations: list[Citation] = Field(description="Una cita por cada afirmación de la respuesta")
    abstained: bool = Field(description="true si los documentos no permiten responder")


ANSWER_INSTRUCTION = """
Responde la pregunta usando solo los documentos entregados.
Cada afirmación debe tener una cita con el identificador del documento y una frase copiada literalmente de él.
Si los documentos no permiten responder, marca abstained=true y no inventes.
""".strip()

corpus_text = "\n\n---\n\n".join(f"[{doc_id}]\n{text}" for doc_id, text in corpus.items())
approx_tokens = len(corpus_text) // 4
if model_ref.startswith("curso:") and approx_tokens > 28_000:
    print(f"⚠ El corpus tiene unos {approx_tokens:,} tokens: no cabe en la ventana del servidor del curso.")


async def respond(case):
    model = create_model(model_ref, config, simulated_output={"answer": "Simulada.", "citations": [], "abstained": True})
    agent = Agent(model, output_type=Answer, instructions=ANSWER_INSTRUCTION, model_settings=SETTINGS, retries=1)
    prompt = f"Documentos:\n\n{corpus_text}\n\n---\n\nPregunta: {case['input']}"
    row = await timed_run(agent, prompt, ref=model_ref, config=config)
    return {"id": case["id"], "task": "answer", **row}


answer_rows = []
for case in answer_cases:
    row = await respond(case)
    answer_rows.append(row)
    print(case["id"], row["status"], row.get("error_type", ""))
    await asyncio.sleep(pause_after(row))

In [ ]:
# Reintentar una vez las consultas que fallaron por el servicio (límite por minuto, saturación, tiempo de espera).
by_case = {case["id"]: case for case in cases}
for rows, run in ((extract_rows, extract), (answer_rows, respond)):
    for index, row in enumerate(rows):
        if row["status"] != "ok":
            await asyncio.sleep(20)
            rows[index] = await run(by_case[row["id"]])
            print("Reintento", row["id"], rows[index]["status"])
print("Fallidas:", [row["id"] for row in extract_rows + answer_rows if row["status"] != "ok"] or "ninguna")

## 5. Métricas

Las funciones calculan las métricas solo con los casos y las salidas guardadas, sin modelo: así el docente puede recalcularlas. Si sus criterios de éxito usan otra métrica, agréguenla aquí.

- **Extracción, por campo:** exactitud (el valor coincide con el esperado), valores inventados (había `null` en la referencia y el modelo puso un valor) y omitidos (había un valor y el modelo puso `null`). Los textos se comparan sin distinguir mayúsculas ni espacios en los extremos.
- **Respuestas:** abstención correcta en las preguntas sin respuesta, respuestas dadas en las que sí la tienen, citas verificables (la frase está literal en el documento citado) y citas correctas (además coincide con una frase esperada: la contiene o es un trozo de ella, como en el laboratorio 3). Si la respuesta es correcta lo revisan ustedes leyendo las respuestas, o con la métrica que hayan definido.

In [ ]:
def normalize(value):
    return value.strip().casefold() if isinstance(value, str) else value


def extract_metrics(cases, outputs, baseline):
    """Exactitud, inventados y omitidos por campo; outputs es {id: salida} y una salida que falta cuenta como error."""
    rows = []
    for field in FIELDS:
        pairs = [(case["expected"].get(field), (outputs.get(case["id"]) or {}).get(field)) for case in cases
                 if field in case["expected"]]
        rows.append({"baseline": baseline, "field": field, "cases": len(pairs),
                     "correct": sum(normalize(exp) == normalize(got) for exp, got in pairs),
                     "invented": sum(exp is None and got is not None for exp, got in pairs),
                     "omitted": sum(exp is not None and got is None for exp, got in pairs)})
    return pd.DataFrame(rows)


def citation_flags(case, citation):
    text = corpus.get(citation["doc_id"], "")
    quote = citation["quote"].strip()
    verifiable = bool(quote) and quote in text
    correct = bool(quote) and any(citation["doc_id"] == e["doc_id"] and (e["quote"] in quote or quote in e["quote"])
                                  for e in case["expected"]["evidence"])
    return verifiable, correct


def answer_metrics(cases, outputs, baseline):
    """Abstención, respuestas dadas y citas; outputs es {id: salida} con answer, citations y abstained."""
    answerable = [case for case in cases if case["expected"]["answerable"]]
    unanswerable = [case for case in cases if not case["expected"]["answerable"]]
    flags = [citation_flags(case, citation) for case in answerable
             for citation in (outputs.get(case["id"]) or {}).get("citations", [])]
    abstained = lambda case: (outputs.get(case["id"]) or {"abstained": True})["abstained"]
    return pd.DataFrame([{
        "baseline": baseline,
        "correct_abstention": f"{sum(abstained(c) for c in unanswerable)}/{len(unanswerable)}",
        "answered": f"{sum(not abstained(c) for c in answerable)}/{len(answerable)}",
        "verifiable_citations": f"{sum(f[0] for f in flags)}/{len(flags)}",
        "correct_citations": f"{sum(f[1] for f in flags)}/{len(flags)}",
        "with_correct_citation": f"{sum(any(citation_flags(c, x)[1] for x in (outputs.get(c['id']) or {}).get('citations', [])) for c in answerable)}/{len(answerable)}",
    }])


def outputs_of(rows):
    return {row["id"]: row["output"] for row in rows if row["status"] == "ok"}


display(pd.concat([
    extract_metrics(extract_cases, outputs_of(extract_rows), f"modelo · {model_ref}"),
    extract_metrics(extract_cases, {case["id"]: {} for case in extract_cases}, "trivial · siempre null"),
]))
display(pd.concat([
    answer_metrics(answer_cases, outputs_of(answer_rows), f"modelo · {model_ref}"),
    answer_metrics(answer_cases, {case["id"]: {"citations": [], "abstained": True} for case in answer_cases},
                   "trivial · siempre sin evidencia"),
]))

In [ ]:
def cost_table(rows, task):
    ok = [row for row in rows if row["status"] == "ok"] if config.mode == "real" else []
    costs = [row.get("estimated_cost_usd") for row in ok]
    return {"task": task, "ok": f"{sum(row['status'] == 'ok' for row in rows)}/{len(rows)}",
            "median_latency_s": round(median(row["latency_ms"] for row in ok) / 1000, 1) if ok and ok[0]["latency_ms"] else None,
            "input_tokens_per_case": round(sum(row.get("input_tokens") or 0 for row in ok) / len(ok)) if ok else None,
            "usd_per_1000_cases": round(1000 * sum(costs) / len(ok), 2) if ok and None not in costs else None}


# Costo y latencia: insumos para comparar alternativas en la sección 4 del Hito 1. En modo simulado quedan vacíos.
display(pd.DataFrame([cost_table(extract_rows, "extract"), cost_table(answer_rows, "answer")]))

In [ ]:
# Casos con errores, para contarlos por categoría de falla en el Hito 1; cambia el id para mirar otros.
def show_case(case_id):
    case, row = by_case[case_id], next(r for r in extract_rows + answer_rows if r["id"] == case_id)
    print(f"{case_id} · {case.get('category', '')}\nEntrada: {str(case['input'])[:400]}\n")
    print("Esperado:", json.dumps(case["expected"], ensure_ascii=False, indent=1))
    print("Obtenido:", json.dumps(row["output"], ensure_ascii=False, indent=1) if row["status"] == "ok" else row["status"])


errors = [case["id"] for case in extract_cases
          if any(normalize(case["expected"].get(f)) != normalize((outputs_of(extract_rows).get(case["id"]) or {}).get(f))
                 for f in FIELDS if f in case["expected"])]
print("Casos de extracción con algún campo distinto:", errors)
if answer_cases:
    show_case(answer_cases[0]["id"])

## 6. Guardar las salidas y recalcular sin modelo

Se guardan en `eval/baseline/`:

- `outputs.jsonl`: una salida por caso, con modelo, tokens, latencia y costo.
- `run.json`: modelo, instrucciones, esquemas y configuración, sin claves.
- `metrics_extract.csv` y `metrics_answer.csv`: las métricas de la sección 5.

La celda siguiente vuelve a calcular las métricas leyendo solo `outputs.jsonl` y sus casos, como hará el docente. Si los números no coinciden con los de la sección 5, algo quedó fuera de los archivos.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
with (OUTPUT / "outputs.jsonl").open("w", encoding="utf-8") as file:
    for row in extract_rows + answer_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")
(OUTPUT / "run.json").write_text(json.dumps({
    "mode": config.mode, "model": model_ref, "settings": SETTINGS, "example_data": USE_EXAMPLE,
    "extract_instruction": EXTRACT_INSTRUCTION, "extract_schema": Record.model_json_schema(),
    "answer_instruction": ANSWER_INSTRUCTION, "answer_schema": Answer.model_json_schema(),
    "corpus_documents": sorted(corpus),
}, ensure_ascii=False, indent=2), encoding="utf-8")

saved = [json.loads(line) for line in (OUTPUT / "outputs.jsonl").read_text(encoding="utf-8").splitlines()]
saved_outputs = {row["id"]: row["output"] for row in saved if row["status"] == "ok"}
extract_table = extract_metrics(extract_cases, saved_outputs, "recalculada desde outputs.jsonl")
answer_table = answer_metrics(answer_cases, saved_outputs, "recalculada desde outputs.jsonl")
extract_table.to_csv(OUTPUT / "metrics_extract.csv", index=False)
answer_table.to_csv(OUTPUT / "metrics_answer.csv", index=False)
display(extract_table)
display(answer_table)

## 7. Descargar y subir al repositorio

La celda siguiente descarga un ZIP con `eval/baseline/`. Descomprímanlo y suban sus archivos a la carpeta `eval/baseline/` del repositorio del equipo desde el navegador (**Add file → Upload files**). Suban también este notebook con sus salidas (**Archivo → Descargar → Descargar .ipynb**): es el código que calcula las métricas. Antes, revisen que no tenga una clave pegada en una celda.

In [ ]:
from datetime import datetime, timezone

archive_name = "hito1-baseline-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
archive_path = Path(shutil.make_archive(str(ROOT / archive_name), "zip", root_dir=ROOT, base_dir="eval/baseline"))
print("Descarga preparada:", archive_path.name)
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab, copia el ZIP desde la ruta indicada.")
else:
    files.download(str(archive_path))